# 5 - Prepare weather data for all ten airports

Verify station idenities, inspect quality codes, and match weather observations to the prediction cutoffs of the selected flights.

In [2]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

NOAA_DIR = PROJECT_ROOT / "data/raw/noaa"

with (PROJECT_ROOT / "outputs/2024_audit/airport_selection.json").open() as f:
    airports = json.load(f)["airports"]

stations = []

for airport in airports:
    for year in [2024, 2025, 2026]:
        path = NOAA_DIR / str(year) / f"noaa_{airport.lower()}_{year}.psv"
        assert path.is_file(), f"Missing: {path.name}"

        row = pd.read_csv(path, sep="|", nrows=1)
        stations.append({
            "airport": airport,
            "year": year,
            "station": row.loc[0, "STATION"],
            "name": row.loc[0, "Station_name"],
            "latitude": row.loc[0, "LATITUDE"],
            "longitude": row.loc[0, "LONGITUDE"],
        })

stations = pd.DataFrame(stations)

assert stations.groupby("airport")["station"].nunique().eq(1).all()
assert stations.groupby("airport")["station"].first().nunique() == 10

display(stations.loc[stations["year"].eq(2024)])
print("All 30 files found")

,airport,year,station,name,latitude,longitude
0,ATL,2024,USW00013874,ATLANTA HARTSFIELD-JACKSON INT,33.6297,-84.4422
3,DFW,2024,USW00003927,DAL-FTW WSCMO AP,32.8975,-97.0219
6,DEN,2024,USW00003017,DENVER INTL AP,39.8467,-104.6561
9,ORD,2024,USW00094846,CHICAGO OHARE INTL AP,41.9603,-87.9317
12,CLT,2024,USW00013881,CHARLOTTE DOUGLAS AP,35.2225,-80.9544
15,LAX,2024,USW00023174,LOS ANGELES INTL AP,33.9381,-118.3867
18,PHX,2024,USW00023183,PHOENIX AP,33.4278,-112.0036
21,LAS,2024,USW00023169,MCCARRAN INTL AP,36.0719,-115.1633
24,SEA,2024,USW00024233,SEATTLE TACOMA AP,47.4447,-122.3144
27,LGA,2024,USW00014732,LAGUARDIA AP,40.7794,-73.8803


All 30 files found


Mapping for methods section

In [3]:
WEATHER_AUDIT = PROJECT_ROOT / "outputs/weather_audit"
WEATHER_AUDIT.mkdir(parents=True, exist_ok=True)

stations.to_csv(WEATHER_AUDIT / "station_mapping.csv", index=False)

## Inspect quality codes across airports

In [5]:
weather_vars = ["temperature", "dew_point_temperature", "wind_speed", "wind_gust", "visibility", "ceiling_height", "precipitation"]
checks = []

for airport in airports:
    for year in [2024, 2025]:
        path = NOAA_DIR / str(year) / f"noaa_{airport.lower()}_{year}.psv"
        df = pd.read_csv(path, sep="|", low_memory=False)

        if year == 2025:
            dates = pd.to_datetime(df["DATE"], utc=True)
            df = df.loc[dates.ge("2025-01-01") & dates.lt("2025-01-02")]

        for var in weather_vars:
            codes = df[[
                f"{var}_Source_Code",
                f"{var}_Quality_Code",
            ]].copy()
            codes.columns = ["source", "quality"]

            for col in codes:
                codes[col] = (
                    codes[col].astype("string").str.strip()
                    .str.replace(r"\.0$", "", regex=True)
                    .fillna("missing")
                )

            counts = codes.value_counts().rename("rows").reset_index()
            checks.append(counts.assign(airport=airport, var=var))

quality_counts = pd.concat(checks, ignore_index=True)

display(quality_counts.groupby(["source", "quality"])["rows"].sum().reset_index())

,source,quality,rows
0,223,1,144642
1,223,2,45
2,223,4,1274
3,343,5,602585
4,343,6,364
5,343,7,56
6,343,A,529
7,343,I,4
8,343,P,44
9,343,U,2


## Clean weather observations

Define a function to clean weather observations based on the quality codes by noaa

In [6]:
def clean_weather(df):
    result = df.copy()

    for var in weather_vars:
        source = pd.to_numeric(df[f"{var}_Source_Code"])
        quality = df[f"{var}_Quality_Code"].astype("string").str.strip().str.replace(r"\.0$", "", regex=True)
        keep = (source.eq(223) & quality.eq("1")) | (source.eq(343) & quality.isin(["1", "5", "A"])).fillna(False)

        result[var] = df[var].where(keep)

    result["precipitation_trace"] = result["precipitation_Measurement_Code"].eq("T").astype("boolean").where(result["precipitation"].notna())
    result["ceiling_unlimited"] = result["ceiling_height"].eq(22000).astype("boolean").where(result["ceiling_height"].notna())
    result["ceiling_height"] = result["ceiling_height"].mask(result["ceiling_unlimited"].fillna(False))
    result["weather_time_utc"] = pd.to_datetime(result["DATE"], utc=True, errors="raise")

    return result.sort_values("weather_time_utc")

## Match weather to all 2024 flights

In [8]:
flights = pd.read_csv(PROJECT_ROOT / "data/processed/flights_2024_top10.csv.gz", dtype={"CRSDepTime": "string"})
flights["prediction_cutoff_utc"] = pd.to_datetime(flights["prediction_cutoff_utc"], utc=True)
flights["flight_id"] = range(len(flights))

weather_features = weather_vars + ["precipitation_trace", "ceiling_unlimited"]
parts = []

for airport in airports:
    yearly = []

    for year in [2024, 2025]:
        path = NOAA_DIR / str(year) / f"noaa_{airport.lower()}_{year}.psv"
        df = pd.read_csv(path, sep="|", low_memory=False)

        expected = stations.loc[stations["airport"].eq(airport), "station"].iloc[0]
        assert df["STATION"].eq(expected).all(), airport

        dates = pd.to_datetime(df["DATE"], utc=True)
        assert dates.notna().all() and dates.dt.year.eq(year).all()

        if year == 2025:
            df = df.loc[dates.lt("2025-01-02")]

        yearly.append(df)

    weather = clean_weather(pd.concat(yearly, ignore_index=True))
    assert weather["weather_time_utc"].notna().all(), airport
    assert not weather["weather_time_utc"].duplicated().any(), airport

    subset = flights.loc[flights["Origin"].eq(airport)]
    joined = pd.merge_asof(
        subset.sort_values("prediction_cutoff_utc"),
        weather[["weather_time_utc"] + weather_features],
        left_on="prediction_cutoff_utc",
        right_on="weather_time_utc",
        direction="backward",
        tolerance=pd.Timedelta("60min"),
    )
    parts.append(joined)
    print(f"{airport}: {joined['weather_time_utc'].notna().mean():.2%} matched")

combined = pd.concat(parts, ignore_index=True).sort_values("flight_id")
combined["weather_age_minutes"] = (
    combined["prediction_cutoff_utc"] - combined["weather_time_utc"]
).dt.total_seconds() / 60

assert len(combined) == len(flights)
assert combined["flight_id"].is_unique
assert combined["weather_age_minutes"].dropna().between(0, 60).all()

ATL: 99.96% matched
DFW: 99.89% matched
DEN: 99.96% matched
ORD: 100.00% matched
CLT: 99.87% matched
LAX: 99.91% matched
PHX: 99.98% matched
LAS: 99.73% matched
SEA: 99.97% matched
LGA: 99.91% matched


## Audit matching coverage

In [9]:
coverage = combined.groupby("Origin").agg(
    flights=("flight_id", "size"),
    matched=("weather_time_utc", "count"),
    median_age_minutes=("weather_age_minutes", "median"),
)
coverage["unmatched"] = coverage["flights"] - coverage["matched"]
coverage["matched_pct"] = 100 * coverage["matched"] / coverage["flights"]

missingness = (combined[weather_features].isna().groupby(combined["Origin"]).mean().mul(100))

display(coverage.round(2))
display(missingness.round(2))

,flights,matched,median_age_minutes,unmatched,matched_pct
Origin,,,,,
ATL,338500,338371,23.0,129,99.96
CLT,214578,214309,23.0,269,99.87
DEN,305160,305053,25.0,107,99.96
DFW,307745,307412,25.0,333,99.89
LAS,187579,187075,26.0,504,99.73
LAX,192471,192299,22.0,172,99.91
LGA,158586,158439,20.0,147,99.91
ORD,276267,276267,23.0,0,100.00
PHX,192373,192333,25.0,40,99.98


,temperature,dew_point_temperature,wind_speed,wind_gust,visibility,ceiling_height,precipitation,precipitation_trace,ceiling_unlimited
Origin,,,,,,,,,
ATL,0.04,0.04,0.04,87.85,0.38,65.14,31.98,31.98,18.11
CLT,0.22,0.22,0.19,90.55,0.16,64.28,35.71,35.71,19.42
DEN,0.04,0.04,0.10,88.75,0.19,69.47,33.91,33.91,15.13
DFW,0.26,0.23,0.21,86.48,0.18,66.22,31.27,31.27,15.69
LAS,0.39,0.38,0.37,85.11,0.31,82.21,37.17,37.17,18.85
LAX,0.11,0.12,0.15,96.27,0.24,65.77,35.51,35.51,14.82
LGA,0.10,0.10,0.15,81.59,0.21,54.96,29.40,29.40,17.23
ORD,0.05,0.03,0.01,81.71,0.04,56.42,32.03,32.03,18.53
PHX,0.18,0.16,0.27,92.21,0.02,83.34,31.98,31.98,13.99


## Save dataset and audit summaries

In [10]:
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

combined.to_csv(PROCESSED_DIR / "flights_weather_2024_top10.csv.gz", index=False)
coverage.to_csv(WEATHER_AUDIT / "all_airports_2024_coverage.csv")
missingness.to_csv(WEATHER_AUDIT / "all_airport_2024_missingness.csv")

quality_counts.to_csv(WEATHER_AUDIT / "all_airport_2024_quality_codes.csv", index=False)

print(f"Saved {len(combined):,} flights with weather and audit summaries")

Saved 2,335,170 flights with weather and audit summaries
